# Digifly Browser Flow Visualizer

Browser-native Phase 2 activity viewer for Docker/JupyterLab sessions. It loads a completed Phase 2 run, overlays activity on SWC morphology using Plotly, and does not require PyVista or a desktop display.


## Resolve Paths

Run a Phase 2 preset first if no completed run exists yet. The default Windows/Docker path is `Phase 1/manc_v1.2.1/export_swc/hemi_runs`.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_phase2_root() -> Path:
    env_root = os.environ.get("DIGIFLY_PHASE2_ROOT", "").strip()
    if env_root:
        return Path(env_root).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "digifly" / "phase2").exists():
            return candidate
        phase2 = candidate / "Phase 2"
        if (phase2 / "digifly" / "phase2").exists():
            return phase2.resolve()
    raise RuntimeError("Could not locate Phase 2 root. Open this notebook from the Digifly Public repo.")


def _latest_run(runs_root: Path) -> Path | None:
    if not runs_root.exists():
        return None
    candidates = []
    for run_dir in runs_root.iterdir():
        if not run_dir.is_dir():
            continue
        if (run_dir / "records.csv").exists():
            candidates.append(run_dir)
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime).resolve()


PHASE2_ROOT = _find_phase2_root()
REPO_ROOT = PHASE2_ROOT.parent
SWC_DIR = Path(os.environ.get("DIGIFLY_SWC_DIR", REPO_ROOT / "Phase 1" / "manc_v1.2.1" / "export_swc")).expanduser().resolve()
RUNS_ROOT = SWC_DIR / "hemi_runs"
RUN_DIR = _latest_run(RUNS_ROOT)

if str(PHASE2_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE2_ROOT))

from digifly.phase2.workbench.browser_visualizer import launch_browser_flow_visualizer, recorded_neuron_ids

print("Phase 2 root:", PHASE2_ROOT)
print("SWC root:", SWC_DIR)
print("Runs root:", RUNS_ROOT)
print("Latest run:", RUN_DIR or "none found")
if RUN_DIR is not None:
    print("Recorded neuron IDs:", recorded_neuron_ids(RUN_DIR))


## Launch The Browser Visualizer

Use the Play button in the Plotly figure to animate the current run. Adjust the widget controls and click **Render Browser Visualizer** to rebuild with different timing or density.


In [ ]:
if RUN_DIR is None:
    raise RuntimeError("No completed Phase 2 run found. Open Digifly_Phase2_Workbench.ipynb and run the Single Neuron Debug preset first.")

browser_app = launch_browser_flow_visualizer(run_dir=RUN_DIR, swc_dir=SWC_DIR)
browser_app
